# 문제 2-4 : 여행 계획 분석기

여행 후기/계획에서 여행지, 기간, 예산, 추천도(1-5점), 주요 활동 리스트를 추출합니다.

- `StructuredOutputParser` 와 `ResponseSchema` 사용
- 5개 필드 정의

### 공통 설정: API Key 로드 + LLM 생성 함수

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


In [2]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, verbose: bool = True, **kwargs) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
        **kwargs,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [3]:
import json
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema  # langchain 1.x: classic 패키지에서 import
from langchain_core.prompts import ChatPromptTemplate

# 출력할 5개 필드를 ResponseSchema 로 정의 (name: 결과 dict 의 키, description: 모델에게 알려줄 설명)
response_schemas = [
    ResponseSchema(name="destination", description="여행지 (도시 또는 국가 이름)"),
    ResponseSchema(name="duration", description="여행 기간 (예: 2박 3일)"),
    ResponseSchema(name="budget", description="여행 예산 또는 사용한 비용 (예: 30만원)"),
    ResponseSchema(name="rating", description="여행 추천도. 1~5 사이의 정수를 문자열로 (예: \"4\")"),
    ResponseSchema(name="activities", description="주요 활동 목록. 각 활동을 '장소 + 활동' 형태의 문자열로 담은 JSON 배열", type="List[string]"),
]

# ResponseSchema 목록으로 파서 생성
parser = StructuredOutputParser.from_response_schemas(response_schemas)

format_instructions = parser.get_format_instructions()
print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"destination": string  // 여행지 (도시 또는 국가 이름)
	"duration": string  // 여행 기간 (예: 2박 3일)
	"budget": string  // 여행 예산 또는 사용한 비용 (예: 30만원)
	"rating": string  // 여행 추천도. 1~5 사이의 정수를 문자열로 (예: "4")
	"activities": List[string]  // 주요 활동 목록. 각 활동을 '장소 + 활동' 형태의 문자열로 담은 JSON 배열
}
```


In [4]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 후기·계획을 분석하는 전문가입니다. 텍스트에서 정보를 추출해 지정된 형식으로만 답하세요. "
               "값은 한국어로 작성하고, 텍스트에 없는 정보는 \"정보 없음\"으로 채우세요.\n{format_instructions}"),
    ("user", "여행 텍스트: {text}"),
]).partial(format_instructions=format_instructions)

llm = get_llm(temperature=0)

# 체인 생성: 프롬프트 → 모델 → Structured 파서
chain = prompt | llm | parser

[LLM 생성] provider=groq | model=openai/gpt-oss-120b


In [5]:
from langchain_core.exceptions import OutputParserException

text = "지난 주에 부산으로 2박 3일 여행을 다녀왔어요. 총 30만원 정도 썼는데 해운대에서 바다구경하고, 자갈치시장에서 회 먹고, 감천문화마을도 구경했어요. 정말 만족스러운 여행이었습니다. 5점 만점에 4점 정도 줄 수 있을 것 같아요."

try:
    result = chain.invoke({"text": text})
    print(type(result))  # dict
    print(json.dumps(result, indent=4, ensure_ascii=False))

    # 필드별 접근
    print(f"여행지: {result['destination']} / 기간: {result['duration']} / 예산: {result['budget']}")
    print(f"추천도: {'⭐' * int(result['rating'])} ({result['rating']}점)")
    print("주요 활동:", ", ".join(result["activities"]))
except OutputParserException as e:
    print(f"파싱 오류: {str(e).splitlines()[0]}")

<class 'dict'>
{
    "destination": "부산",
    "duration": "2박 3일",
    "budget": "30만원",
    "rating": "4",
    "activities": [
        "해운대 + 바다구경",
        "자갈치시장 + 회 먹기",
        "감천문화마을 + 구경"
    ]
}
여행지: 부산 / 기간: 2박 3일 / 예산: 30만원
추천도: ⭐⭐⭐⭐ (4점)
주요 활동: 해운대 + 바다구경, 자갈치시장 + 회 먹기, 감천문화마을 + 구경


In [6]:
# 다른 여행 텍스트로도 테스트 (후기 + 계획)
texts = [
    "다음 달에 친구랑 제주도로 3박 4일 여행 갈 계획이에요. 예산은 1인당 50만원 잡았고, 한라산 등반, 우도 자전거 일주, 흑돼지 맛집 투어를 하려고 해요. 기대 만점이라 추천도는 5점!",
    "작년 겨울 일본 삿포로에 4박 5일 다녀왔는데 약 120만원 들었어요. 눈축제 구경하고 오타루 운하 산책하고 온천도 갔어요. 날씨가 너무 추워서 고생했지만 3점은 줄게요.",
]

for t in texts:
    try:
        print(json.dumps(chain.invoke({"text": t}), indent=4, ensure_ascii=False))
    except OutputParserException as e:
        print(f"파싱 오류: {str(e).splitlines()[0]}")
    print("-" * 50)

{
    "destination": "제주도",
    "duration": "3박 4일",
    "budget": "1인당 50만원",
    "rating": "5",
    "activities": [
        "한라산 등반",
        "우도 자전거 일주",
        "흑돼지 맛집 투어"
    ]
}
--------------------------------------------------
{
    "destination": "삿포로",
    "duration": "4박 5일",
    "budget": "120만원",
    "rating": "3",
    "activities": [
        "삿포로 눈축제 구경",
        "오타루 운하 산책",
        "삿포로 온천"
    ]
}
--------------------------------------------------
